# Crime Geography and Maps

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

import pandas as pd
from crime_snapshot import load_crime_snapshot
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
import plotly.graph_objects as go
from spd_dashboard_data import load_mcpp_boundaries
warnings.filterwarnings('ignore')

df, metadata = load_crime_snapshot(
    PROJECT_ROOT / "data" / "processed" / "crime"
)

display(df.head())

CRIME_OUTPUT_DIR =  PROJECT_ROOT / "reports" / "crime"

TIME_COLUMN = "offense_date"
EVENT_ID_COLUMN = "offense_id"
CATEGORY_COLUMN = "offense_category"
NEIGHBORHOOD_COLUMN = "neighborhood"
LAT_COL = "latitude"
LON_COL = "longitude"

PLOTLY_TEMPLATE = "plotly_dark"
PLOT_BG = "#545455"
PAPER_BG = "#111111"
PLOTLY_MAP_STYLE = "carto-darkmatter"
PLOTLY_SEATTLE_CENTER = {"lat": 47.6062, "lon": -122.3321}
mcpp_boundaries = load_mcpp_boundaries()


In [ ]:
mcpp_boundaries.columns.tolist()
boundary_name_column = "mcpp_neighborhood_display"
feature_id_column = "plot_feature_id"

mcpp_boundaries[[
        "objectid",
        "mcpp_neighborhood",
        "mcpp_precinct",
        "plot_feature_id",
        "mcpp_neighborhood_display",]
    ].head(20)

In [ ]:
map_test_df = df.copy()

map_test_df[TIME_COLUMN] = pd.to_datetime(
    map_test_df[TIME_COLUMN],
    errors="coerce",
)

map_test_df[CATEGORY_COLUMN] = (
    map_test_df[CATEGORY_COLUMN]
    .astype("string")
    .str.strip()
    .str.lower()
)

map_test_df[NEIGHBORHOOD_COLUMN] = (
    map_test_df[NEIGHBORHOOD_COLUMN]
    .astype("string")
    .str.strip()
)

map_test_df[LAT_COL] = pd.to_numeric(
    map_test_df[LAT_COL],
    errors="coerce",
)

map_test_df[LON_COL] = pd.to_numeric(
    map_test_df[LON_COL],
    errors="coerce",
)

map_test_df = map_test_df.dropna(
    subset=[
        TIME_COLUMN,
        EVENT_ID_COLUMN,
        CATEGORY_COLUMN,
    ]
)

map_test_df.shape

In [ ]:
available_categories = (
    map_test_df[CATEGORY_COLUMN]
    .dropna()
    .sort_values()
    .unique()
    .tolist()
)

available_categories

selected_categories = [
    "all other",
    "property crime",
    "violent crime",
]
filtered_crime_df = map_test_df[
    map_test_df[CATEGORY_COLUMN].isin(selected_categories)
].copy()

filtered_crime_df.shape

In [ ]:
filtered_crime_df["has_neighborhood"] = (
    filtered_crime_df[NEIGHBORHOOD_COLUMN].notna()
    & (filtered_crime_df[NEIGHBORHOOD_COLUMN].astype("string").str.strip() != "")
)

filtered_crime_df["has_coordinates"] = (
    filtered_crime_df[LAT_COL].notna()
    & filtered_crime_df[LON_COL].notna()
)

mapping_coverage = pd.DataFrame(
    {
        "metric": [
            "total records",
            "records with neighborhood",
            "records with coordinates",
        ],
        "count": [
            len(filtered_crime_df),
            filtered_crime_df["has_neighborhood"].sum(),
            filtered_crime_df["has_coordinates"].sum(),
        ],
    }
)

mapping_coverage["percent"] = (
    mapping_coverage["count"] / len(filtered_crime_df) * 100
).round(2)

mapping_coverage

In [ ]:
crime_by_neighborhood = (
    filtered_crime_df[filtered_crime_df["has_neighborhood"]]
    .groupby(NEIGHBORHOOD_COLUMN)
    .agg(
        reported_offenses=(EVENT_ID_COLUMN, "nunique"),
        records=(EVENT_ID_COLUMN, "size"),
    )
    .reset_index()
    .sort_values("reported_offenses", ascending=False)
)

crime_by_neighborhood.head(30)

In [ ]:
def normalize_neighborhood_name(series: pd.Series) -> pd.Series:
    return (
        series
        .astype("string")
        .str.strip()
        .str.lower()
        .str.replace("&", "and", regex=False)
        .str.replace(r"\s+", " ", regex=True)
    )
    boundary_name_column = "mcpp_neighborhood_display"
feature_id_column = "plot_feature_id"

crime_by_neighborhood = crime_by_neighborhood.copy()
mcpp_boundaries = mcpp_boundaries.copy()

crime_by_neighborhood["neighborhood_norm"] = normalize_neighborhood_name(
    crime_by_neighborhood[NEIGHBORHOOD_COLUMN]
)

mcpp_boundaries["neighborhood_norm"] = normalize_neighborhood_name(
    mcpp_boundaries[boundary_name_column]
)

crime_neighborhoods = set(
    crime_by_neighborhood["neighborhood_norm"]
    .dropna()
    .astype(str)
    .str.strip()
)

boundary_neighborhoods = set(
    mcpp_boundaries["neighborhood_norm"]
    .dropna()
    .astype(str)
    .str.strip()
)

matched_neighborhoods = crime_neighborhoods & boundary_neighborhoods
crime_only_neighborhoods = sorted(crime_neighborhoods - boundary_neighborhoods)
boundary_only_neighborhoods = sorted(boundary_neighborhoods - crime_neighborhoods)

print("Crime neighborhoods:", len(crime_neighborhoods))
print("Boundary neighborhoods:", len(boundary_neighborhoods))
print("Matched neighborhoods:", len(matched_neighborhoods))
print("Crime-only neighborhoods:", len(crime_only_neighborhoods))

crime_only_neighborhoods[:50]

In [ ]:
choropleth_df = mcpp_boundaries.merge(
    crime_by_neighborhood,
    on="neighborhood_norm",
    how="left",
    suffixes=("_boundary", "_crime"),
)

choropleth_df["reported_offenses"] = (
    choropleth_df["reported_offenses"]
    .fillna(0)
    .astype(int)
)

choropleth_df[
    [
        boundary_name_column,
        "reported_offenses",
    ]
].sort_values("reported_offenses", ascending=False).head(20)

fig = go.Figure()

fig.add_trace(
    go.Choroplethmapbox(
        geojson=choropleth_df.__geo_interface__,
        locations=choropleth_df[feature_id_column],
        z=choropleth_df["reported_offenses"],
        featureidkey=f"properties.{feature_id_column}",
        colorscale="Reds",
        marker_opacity=0.65,
        marker_line_width=1,
        marker_line_color="rgba(255,255,255,0.35)",
        colorbar=dict(
            title="Reported<br>offenses",
        ),
        customdata=choropleth_df[[boundary_name_column]],
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Reported offenses: %{z:,}<extra></extra>"
        ),
        name="Reported offenses",
    )
)

fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title=(
        "Reported Crime by Neighborhood"
        "<br><sup>Test choropleth using normalized crime neighborhood names</sup>"
    ),
    mapbox=dict(
        style=PLOTLY_MAP_STYLE,
        center=PLOTLY_SEATTLE_CENTER,
        zoom=10.2,
    ),
    plot_bgcolor=PLOT_BG,
    paper_bgcolor=PAPER_BG,
    height=750,
    margin=dict(l=0, r=0, t=80, b=0),
)

fig.show()

In [ ]:
# Test combined crime choropleth + point map

import plotly.graph_objects as go

selected_categories = [
    "all other",
    "property crime",
    "violent crime",
]

category_color_map = {
    "all other": "#9CA3AF",
    "property crime": "#27AE60",
    "violent crime": "#EB5757",
}

latest_available_day = map_test_df[TIME_COLUMN].max().normalize()
map_start_day = latest_available_day - pd.Timedelta(days=29)
map_end_day = latest_available_day

print("Map date window:")
print(map_start_day, "to", map_end_day)

map_window_df = map_test_df[
    (map_test_df[TIME_COLUMN] >= map_start_day)
    & (map_test_df[TIME_COLUMN] <= map_end_day)
    & (map_test_df[CATEGORY_COLUMN].isin(selected_categories))
].copy()

map_window_df["has_neighborhood"] = (
    map_window_df[NEIGHBORHOOD_COLUMN].notna()
    & (map_window_df[NEIGHBORHOOD_COLUMN].astype("string").str.strip() != "")
)

map_window_df["has_coordinates"] = (
    map_window_df[LAT_COL].notna()
    & map_window_df[LON_COL].notna()
)

print("Total records:", len(map_window_df))
print("Records with neighborhood:", map_window_df["has_neighborhood"].sum())
print("Records with coordinates:", map_window_df["has_coordinates"].sum())

crime_by_neighborhood = (
    map_window_df[map_window_df["has_neighborhood"]]
    .groupby(NEIGHBORHOOD_COLUMN)
    .agg(
        reported_offenses=(EVENT_ID_COLUMN, "nunique"),
        records=(EVENT_ID_COLUMN, "size"),
    )
    .reset_index()
)

crime_by_neighborhood["neighborhood_norm"] = normalize_neighborhood_name(
    crime_by_neighborhood[NEIGHBORHOOD_COLUMN]
)

mcpp_boundaries = mcpp_boundaries.copy()

if "neighborhood_norm" not in mcpp_boundaries.columns:
    mcpp_boundaries["neighborhood_norm"] = normalize_neighborhood_name(
        mcpp_boundaries[boundary_name_column]
    )

choropleth_df = mcpp_boundaries.merge(
    crime_by_neighborhood,
    on="neighborhood_norm",
    how="left",
    suffixes=("_boundary", "_crime"),
)

choropleth_df["reported_offenses"] = (
    choropleth_df["reported_offenses"]
    .fillna(0)
    .astype(int)
)

choropleth_df["records"] = (
    choropleth_df["records"]
    .fillna(0)
    .astype(int)
)

choropleth_df[
    [
        boundary_name_column,
        "reported_offenses",
        "records",
    ]
].sort_values("reported_offenses", ascending=False).head(20)

point_df = map_window_df[map_window_df["has_coordinates"]].copy()

# Basic Seattle-ish bounding box filter
point_df = point_df[
    point_df[LAT_COL].between(47.45, 47.75)
    & point_df[LON_COL].between(-122.45, -122.20)
].copy()

# Keep notebook rendering manageable.
# If this is small, this line does nothing.
max_points = 5000

if len(point_df) > max_points:
    point_df = point_df.sample(
        n=max_points,
        random_state=42,
    )

point_df["offense_date_label"] = point_df[TIME_COLUMN].dt.strftime("%b %d, %Y")

point_df.shape

fig = go.Figure()

# Choropleth base layer
fig.add_trace(
    go.Choroplethmapbox(
        geojson=choropleth_df.__geo_interface__,
        locations=choropleth_df[feature_id_column],
        z=choropleth_df["reported_offenses"],
        featureidkey=f"properties.{feature_id_column}",
        colorscale="Reds",
        marker_opacity=0.65,
        marker_line_width=1,
        marker_line_color="rgba(255,255,255,0.35)",
        colorbar=dict(
            title="Reported<br>offenses",
        ),
        customdata=choropleth_df[
            [
                boundary_name_column,
                "reported_offenses",
            ]
        ],
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Reported offenses: %{customdata[1]:,}<extra></extra>"
        ),
        name="Neighborhood reported offenses",
        showlegend=False,
    )
)

# Point overlay, one trace per category so the legend is readable
for category in selected_categories:
    category_point_df = point_df[
        point_df[CATEGORY_COLUMN] == category
    ].copy()

    if category_point_df.empty:
        continue

    fig.add_trace(
        go.Scattermapbox(
            lat=category_point_df[LAT_COL],
            lon=category_point_df[LON_COL],
            mode="markers",
            marker=dict(
                size=6,
                opacity=0.55,
                color=category_color_map.get(category, "#DDDDDD"),
            ),
            customdata=category_point_df[
                [
                    EVENT_ID_COLUMN,
                    NEIGHBORHOOD_COLUMN,
                    "offense_sub_category",
                    "offense_date_label",
                ]
            ],
            hovertemplate=(
                "<b>" + category.title() + "</b><br>"
                "Sub-category: %{customdata[2]}<br>"
                "Neighborhood: %{customdata[1]}<br>"
                "Date: %{customdata[3]}<br>"
                "Offense ID: %{customdata[0]}<extra></extra>"
            ),
            name=f"{category.title()} points",
        )
    )

fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title=(
        "Reported Crime by Neighborhood with Exact-Coordinate Points"
        f"<br><sup>{map_start_day.date()} to {map_end_day.date()}</sup>"
    ),
    mapbox=dict(
        style=PLOTLY_MAP_STYLE,
        center=PLOTLY_SEATTLE_CENTER,
        zoom=10.2,
    ),
    plot_bgcolor=PLOT_BG,
    paper_bgcolor=PAPER_BG,
    height=750,
    margin=dict(l=0, r=0, t=90, b=0),
    legend=dict(
        title="Point layer",
        orientation="v",
        yanchor="middle",
        y=0.5,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(0,0,0,0.45)",
    ),
)

fig.show()